# Gemma-4 Benchmark: Base Model vs. Fine-Tune

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilalAbic/math-toolcall-tr/blob/main/notebooks/benchmark_base_vs_finetune.ipynb)

İki modeli, üç benchmark ile karşılaştırır:

| Model | Ne |
|---|---|
| **Base** | `unsloth/gemma-4-E4B-it` — hiç eğitilmemiş hâli |
| **Fine-tune** | `bilalabic/gemma_4_math-toolcall-tr_lora` — bizim LoRA adaptörümüz |

| Benchmark | Ölçtüğü şey | Neden |
|---|---|---|
| **1. Türkçe MMLU** | Genel bilgi (çoktan seçmeli) | Fine-tune genel yeteneği **bozdu mu?** (catastrophic forgetting) |
| **2. Matematik Tool-Call** | Doğru aracı çağırma + gerekmiyorsa çağırmama | **Asıl hedefimiz** — iyileşme burada olmalı |
| **3. GSM8K** | Serbest matematik akıl yürütme | Sektör standardı. Matematik yeteneği **korundu mu?** |

### Üç tasarım kararı

**1. Ollama değil Unsloth.** Fine-tune modelimiz GGUF olarak Ollama'da yok, HF'de LoRA
adaptörü olarak duruyor. Unsloth ikisini de doğrudan yükleyebiliyor.

**2. Tek yükleme, adaptörü aç/kapa.** LoRA = base ağırlıklar + küçük bir ek. Adaptörü
`disable_adapter()` ile kapatınca elimizde **tam olarak base model** kalır. Bu hem ~10 GB
bellek kazandırır hem de iki modelin **aynı kuantizasyonla** çalışmasını garanti eder —
yoksa karşılaştırma adil olmaz.

**3. Gerçek held-out test.** Model veri setinin **757 örneklik** hâliyle eğitildi; veri
seti şimdi **1.328**. Yani **571 örneği model hiç görmedi**. Matematik benchmark'ı bunun
üzerine kurulur — ezber değil, genelleme ölçülür.

Bu sınırı *varsayımla* değil **zaman damgasıyla** belirliyoruz ve notebook her
çalıştırmada yeniden doğruluyor (bkz. Bölüm 6).

> ⏱️ **Süre:** Üretim toplu (batched) yapılır — tek tek üretime göre 5-8x hızlı.
> MMLU 250 + matematik 150 + GSM8K 150, iki model için ≈ **1-2 saat** (GPU'ya bağlı).
> Aceleci isen ilk hücredeki soru sayılarını düşür.

## 1. Kurulum

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec sentence-transformers pandas matplotlib
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Ayarlar

Buradaki sayıları değiştirerek testin kapsamını ve süresini ayarlarsın.

In [ ]:
# ============================ AYARLAR ============================
FINETUNE_REPO = "bilalabic/gemma_4_math-toolcall-tr_lora"   # LoRA adaptorumuz (base'i icinde tasir)
DATASET_REPO  = "bilalabic/math-toolcall-tr"                # kendi veri setimiz

# Kac soru test edilsin? Dusuk tut -> hizli calisir; yuksek tut -> sonuc daha guvenilir.
# MMLU tam veri seti binlerce soru; 250 makul bir denge.
MMLU_SORU_SAYISI = 250
MATEMATIK_ORNEK_SAYISI = 150
GSM8K_SORU_SAYISI = 150      # GSM8K test setinde 1319 soru var; 150 makul bir orneklem

# Model 757 ornekle egitildi. Bu indeksten SONRAKI ornekleri model HIC gormedi.
# Matematik benchmark'i sadece bu gorulmemis kisim uzerinde calisir.
EGITIMDE_KULLANILAN = 757

MAX_SEQ_LENGTH = 2048   # modele verilebilecek en uzun girdi (token). Egitimde de 2048 kullanildi.
SEED = 42               # tekrarlanabilirlik icin sabit tohum

# Kac promptu AYNI ANDA isleyelim? En kritik hiz ayari.
# Tek tek uretim GPU'yu bos birakir; toplu uretim 5-8x hizlandirir.
# Bellek yetmezse kod otomatik yariya duser, ama bastan dusuk vermek daha hizli baslatir:
#   A100 80GB -> 16-24  |  L4 24GB -> 8  |  T4 16GB -> 4
BATCH_BOYUTU = 8
# =================================================================

import torch, time, json, re, gc
import pandas as pd
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Runtime > GPU sec!")

## 3. Modeli yükle

Tek bir yükleme yapıyoruz: LoRA adaptörü **base modeli de beraberinde** getirir.
Sonra adaptörü açıp kapatarak iki modeli de test edeceğiz.

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name     = FINETUNE_REPO,
    # dtype=None -> donanima gore otomatik secer (A100'de bfloat16, T4'te float16).
    dtype          = None,
    # Modelin isleyebilecegi en uzun dizi. Uzun tutmak bellek yer; 2048 bizim egitimle ayni.
    max_seq_length = MAX_SEQ_LENGTH,
    # 4-bit kuantizasyon: agirliklari 4 bite sikistirir. ~4x az bellek, cok az dogruluk kaybi.
    # Egitim de 4-bit yapildigi icin test de 4-bit olmali -> adil karsilastirma.
    load_in_4bit   = True,
)

# Modeli degerlendirme moduna al: dropout gibi egitime ozel katmanlari kapatir.
model.eval()
print("Model yuklendi. Adaptor aktif mi:", hasattr(model, "disable_adapter"))

## 4. Üretim fonksiyonu — parametreler ne işe yarıyor?

Benchmark'ta **belirlenimci (deterministic)** üretim kullanıyoruz. Sebep: aynı soruya
her çalıştırmada aynı cevabı vermeli ki iki model adil karşılaştırılsın ve sonuç
tekrarlanabilir olsun.

| Parametre | Ne yapar | Benchmark'ta neden böyle |
|---|---|---|
| `do_sample=False` | Rastgelelik yok, her adımda **en olası** token seçilir (greedy) | Aynı girdi → aynı çıktı. Ölçüm gürültüsü sıfır |
| `temperature` | Olasılıkları yayar/keskinleştirir. Yüksek = yaratıcı, düşük = tutarlı | `do_sample=False` iken **etkisiz**, o yüzden göndermiyoruz |
| `top_p` / `top_k` | Örnekleme havuzunu daraltır | Yine sadece `do_sample=True` iken anlamlı |
| `max_new_tokens` | En fazla kaç token üretilecek | MMLU'da kısa (tek harf yeter), tool-call'da uzun (JSON + açıklama) |
| `use_cache=True` | Önceki hesapları saklar (KV cache) | Üretimi belirgin biçimde hızlandırır |

> **Not:** Gemma-4'ün önerdiği `temperature=1.0, top_p=0.95, top_k=64` ayarları **sohbet**
> içindir. Benchmark'ta rastgelelik istemeyiz — bu yüzden greedy kullanıyoruz.

In [ ]:
@torch.no_grad()   # gradyan hesabini kapatir -> daha hizli, cok daha az bellek
def uret(mesaj: str, max_new_tokens: int = 64) -> str:
    """Modele bir kullanici mesaji verir, urettigi metni dondurur."""
    mesajlar = [{"role": "user", "content": [{"type": "text", "text": mesaj}]}]

    # Chat template'i uygular: mesaji modelin bekledigi <start_of_turn> formatina cevirir.
    girdi = tokenizer.apply_chat_template(
        mesajlar,
        add_generation_prompt = True,   # sona "model sirasi" isareti koyar -> model cevap yazmaya baslar
        tokenize    = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    cikti = model.generate(
        **girdi,
        max_new_tokens = max_new_tokens,
        do_sample      = False,   # greedy -> belirlenimci
        use_cache      = True,
    )

    # Sadece YENI uretilen kismi al (girdi promptunu kes).
    yeni_tokenlar = cikti[0][girdi["input_ids"].shape[1]:]
    return tokenizer.decode(yeni_tokenlar, skip_special_tokens=True).strip()


def ilerleme(guncel, toplam, uzunluk=30):
    dolu = int(uzunluk * guncel / toplam)
    return f"[{'#' * dolu}{'-' * (uzunluk - dolu)}] {100 * guncel / toplam:5.1f}%"


# ---------------- TOPLU URETIM (asil hiz kazanci burada) ----------------
# Tek tek uretimde GPU'nun buyuk kismi bos bekler. Ayni anda N promptu
# isleyince ayni sure icinde N kat is yapilir.
#
# Iki teknik zorunluluk:
#  1) padding_side="left"  -> decoder-only modellerde uretim SONDAN devam eder.
#     Sagdan doldurursak model padding tokenlarinin ustune yazar, cikti bozulur.
#  2) attention_mask       -> modelin padding'i gormezden gelmesini saglar.
# Gemma-4'te "tokenizer" bir Processor olabilir; gercek tokenizer icinde durur.
# Ayari yanlis nesneye uygularsak SESSIZCE bozuk cikti aliriz - o yuzden ikisini de ayarla.
_tok = getattr(tokenizer, "tokenizer", tokenizer)
for _t in {id(tokenizer): tokenizer, id(_tok): _tok}.values():
    try:
        _t.padding_side = "left"
        if getattr(_t, "pad_token", None) is None and getattr(_t, "eos_token", None):
            _t.pad_token = _t.eos_token
    except AttributeError:
        pass

PAD_ID = getattr(_tok, "pad_token_id", None) or getattr(_tok, "eos_token_id", None)
print("padding_side :", getattr(_tok, "padding_side", "?"), "(left olmali)")
print("pad_token_id :", PAD_ID)


@torch.no_grad()
def uret_toplu(mesajlar: list, max_new_tokens: int = 256, batch: int = None) -> list:
    """Coklu promptu gruplar halinde isler. Bellek yetmezse grubu kucultup devam eder."""
    batch = batch or BATCH_BOYUTU
    sonuclar, i, basla = [], 0, time.time()

    while i < len(mesajlar):
        parca = mesajlar[i:i + batch]

        # Chat template'i metin olarak uygula; <bos>'u kaldiriyoruz cunku
        # tokenizer bir tane daha ekleyecek (cift <bos> modeli sasirtir).
        metinler = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": [{"type": "text", "text": m}]}],
                add_generation_prompt=True, tokenize=False,
            ).removeprefix("<bos>")
            for m in parca
        ]

        try:
            # DIKKAT: Gemma-4'un "tokenizer"i aslinda bir Processor ve imzasi
            #   __call__(self, images, text, audio, videos, **kwargs)
            # yani ILK konumsal argüman 'images'. Metni mutlaka text= ile ver,
            # yoksa metin images'a gider ve "NoneType is not subscriptable" alirsin.
            girdi = tokenizer(text=metinler, return_tensors="pt", padding=True,
                              truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
            cikti = model.generate(
                **girdi,
                max_new_tokens = max_new_tokens,
                do_sample      = False,          # greedy -> belirlenimci
                use_cache      = True,
                pad_token_id   = PAD_ID,
            )
        except torch.cuda.OutOfMemoryError:
            if batch == 1:
                raise
            batch = max(1, batch // 2)
            torch.cuda.empty_cache()
            print(f"\n  bellek yetmedi -> batch {batch}'e dusuruldu")
            continue                              # ayni parcayi kucuk batch ile tekrar dene

        # Sol dolgu sayesinde tum dizilerin girdi uzunlugu ayni -> tek kesim yeterli
        girdi_uzunlugu = girdi["input_ids"].shape[1]
        for j in range(len(parca)):
            yeni = cikti[j][girdi_uzunlugu:]
            sonuclar.append(tokenizer.decode(yeni, skip_special_tokens=True).strip())

        i += len(parca)
        gecen = time.time() - basla
        kalan = gecen / i * (len(mesajlar) - i)
        print(f"\r  {ilerleme(i, len(mesajlar))} {gecen:.0f}s gecti, ~{kalan:.0f}s kaldi", end="")

    print()
    return sonuclar


# Adaptoru kapatip acmayi kolaylastiran yardimci.
from contextlib import contextmanager

# Guvenlik kontrolu: adaptor kapatilamiyorsa iki model de AYNI olur ve
# karsilastirma sessizce anlamsizlasir. Bunu bastan yakalayalim.
assert hasattr(model, "disable_adapter"), (
    "Model bir PeftModel degil - adaptor kapatilamiyor.\n"
    "Cozum: base'i ayrica yukle ->\n"
    "  base_model, _ = FastModel.from_pretrained('unsloth/gemma-4-E4B-it',\n"
    "                      max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)"
)

@contextmanager
def model_olarak(hangisi: str):
    """'base' -> LoRA kapali (saf temel model) | 'finetune' -> LoRA acik."""
    if hangisi == "base":
        with model.disable_adapter():   # adaptor agirliklarini gecici olarak devre disi birakir
            yield
    else:
        yield

# Hizli duman testi: iki model FARKLI cikti veriyor mu?
# Ikisi de birebir ayni ciktiyi veriyorsa adaptor gercekten devrede degil demektir.
_ciktilar = {}
for ad in ("base", "finetune"):
    with model_olarak(ad):
        _ciktilar[ad] = uret("12'nin asal çarpanlarını bul.", max_new_tokens=80)
    print(f"[{ad:9}] {_ciktilar[ad][:150]}\n")

if _ciktilar["base"] == _ciktilar["finetune"]:
    print("UYARI: iki cikti birebir ayni. Adaptor devrede olmayabilir - kontrol et!")
else:
    print("OK: adaptor davranisi degistiriyor, karsilastirma anlamli.")


# --- TOPLU URETIM DOGRULUK TESTI ---
# Toplu uretim, tek tek uretimle AYNI sonucu vermeli. Vermiyorsa sol dolgu (left
# padding) dogru uygulanmamis demektir ve TUM benchmark sonuclari cop olur.
# Bu yuzden benchmark'lara baslamadan once kucuk bir karsilastirma yapiyoruz.
_test = ["7 ile 13'un carpimi kac?", "100'un yarisinin yarisi kactir?"]
_tekil = [uret(p, max_new_tokens=40) for p in _test]
_toplu = uret_toplu(_test, max_new_tokens=40, batch=2)

if _tekil == _toplu:
    print("\nOK: toplu uretim = tek tek uretim. Sol dolgu dogru calisiyor.")
else:
    print("\nUYARI! Toplu ve tekil cikti FARKLI - sol dolgu hatali olabilir:")
    for a, b in zip(_tekil, _toplu):
        if a != b:
            print(f"   tekil: {a[:90]}")
            print(f"   toplu: {b[:90]}")

## 5. Benchmark 1 — Türkçe MMLU

Çoktan seçmeli genel bilgi testi. Burada **iyileşme beklemiyoruz** — beklentimiz
fine-tune'un genel yeteneği *bozmamış* olması. Skor ciddi düştüyse model dar bir alana
aşırı uyum sağlamış (catastrophic forgetting) demektir.

Cevap kontrolü üç aşamalı: (1) doğrudan harf eşleşmesi, (2) "A)" gibi ön ekleri ayıklama,
(3) model harf yerine cümle yazdıysa **anlamsal benzerlik** ile en yakın şıkkı bulma.

In [ ]:
from sentence_transformers import SentenceTransformer

# Model harf yerine "cevap 12'dir" gibi yazarsa, hangi sikka en yakin oldugunu bulmak icin.
benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

HARFLER = ['A', 'B', 'C', 'D', 'E']

def cevap_dogru_mu(dogru_index: int, verilen: str, secenekler: list) -> bool:
    dogru_harf = HARFLER[dogru_index]
    verilen = verilen.upper().strip()

    # 1) Tam harf eslesmesi
    if verilen == dogru_harf:
        return True

    # 2) "A)", "A:", "A -" gibi kaliplarin ilk harfini al
    if len(verilen) > 1 and verilen[1] in [" ", ":", ")", "=", "-", ".", ","]:
        return verilen[0] == dogru_harf

    # 3) Serbest metin -> anlamsal olarak en yakin sikki bul
    if not verilen:
        return False
    e_cevap = benzerlik_modeli.encode([verilen])
    e_secenek = benzerlik_modeli.encode(secenekler)
    puanlar = benzerlik_modeli.similarity(e_cevap, e_secenek).tolist()[0]
    return puanlar.index(max(puanlar)) == dogru_index


# Veri setini yukle (alibayram'in Turkce MMLU calismasi)
mmlu = pd.read_parquet(
    "hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet"
)
# Sabit tohumla karistir -> her calistirmada AYNI alt kume secilir (tekrarlanabilirlik)
mmlu = mmlu.sample(n=min(MMLU_SORU_SAYISI, len(mmlu)), random_state=SEED).reset_index(drop=True)
print(f"MMLU: {len(mmlu)} soru | bolum sayisi: {mmlu['bolum'].nunique()}")

In [ ]:
# Promptlari bir kere kur (iki model de AYNI promptlari gorur)
mmlu_promptlar = []
for i in range(len(mmlu)):
    satir = mmlu.iloc[i]
    metin = satir['soru'] + "\n"
    for j, secenek in enumerate(satir['secenekler']):
        metin += f"{HARFLER[j]}: {secenek}\n"
    mmlu_promptlar.append(
        "Sana soru ve seçenekleri veriyorum. Sadece hangi seçeneğin doğru "
        "cevap olduğunu yaz. Örneğin 'A' veya 'B' gibi. Açıklama yapma!\n"
        "Soru: " + metin
    )


def mmlu_calistir(model_adi: str) -> dict:
    basla = time.time()
    print(f"[{model_adi}] MMLU uretiliyor...")

    # max_new_tokens=16: tek harf bekliyoruz, uzun uretim sadece zaman kaybi
    with model_olarak(model_adi):
        cevaplar = uret_toplu(mmlu_promptlar, max_new_tokens=16)

    # Puanlama (GPU gerektirmez, hizli)
    dogru = 0
    bolum_dogru, bolum_toplam = {}, {}
    kayitlar = []
    for i, cevap in enumerate(cevaplar):
        satir = mmlu.iloc[i]
        sonuc = cevap_dogru_mu(satir['cevap'], cevap, list(satir['secenekler']))
        bolum = satir['bolum']
        bolum_toplam[bolum] = bolum_toplam.get(bolum, 0) + 1
        if sonuc:
            dogru += 1
            bolum_dogru[bolum] = bolum_dogru.get(bolum, 0) + 1
        kayitlar.append({"soru_no": i, "bolum": bolum, "model": model_adi,
                         "cevap": cevap, "dogru_mu": sonuc})

    print(f"[{model_adi}] dogru {dogru}/{len(mmlu)} = {100*dogru/len(mmlu):.1f}%")

    return {
        "model": model_adi,
        "dogru": dogru,
        "toplam": len(mmlu),
        "basari": round(100 * dogru / len(mmlu), 2),
        "sure_sn": round(time.time() - basla, 1),
        "bolum_basari": {b: round(100 * bolum_dogru.get(b, 0) / t, 1) for b, t in bolum_toplam.items()},
        "kayitlar": kayitlar,
    }


mmlu_sonuc = {ad: mmlu_calistir(ad) for ad in ("base", "finetune")}
for ad, s in mmlu_sonuc.items():
    print(f"{ad:9} -> %{s['basari']}  ({s['dogru']}/{s['toplam']}, {s['sure_sn']}s)")

## 6. Benchmark 2 — Matematik Tool-Call

**Asıl ölçmek istediğimiz.** Modele matematik fonksiyonları sunuyoruz ve bakıyoruz:

| Metrik | Ne ölçer |
|---|---|
| **Format geçerliliği** | `<tool_call>{...}</tool_call>` düzgün JSON mu? |
| **Araç seçimi** | Doğru fonksiyonu mu çağırdı? |
| **Çekimserlik (abstain)** | Araç *gerekmediğinde* çağırmamayı biliyor mu? |
| **Genel doğruluk** | Yukarıdakilerin birleşimi |

### ⚖️ Adil karşılaştırma: iki modele de araçları gösteriyoruz

Kritik nokta. Modele **sadece soruyu** gönderirsek base model hangi araçların var
olduğunu ve hangi formatta cevap vermesi gerektiğini bilemez — hiç `<tool_call>`
üretmez ve yalnızca "araç gerekmiyor" örneklerinden puan toplar. O zaman skoru
modelin yeteneğini değil, sadece *formatı bilmediğini* ölçer; fine-tune otomatik
kazanır ve karşılaştırma anlamsızlaşır.

Bu yüzden prompt'a **araç şemalarını ve format talimatını** koyuyoruz — ikisine de
aynısını. Böylece base model de gerçekten deneyebilir; fine-tune kazanırsa bunu
*daha iyi olduğu için* kazanır, tek seçenek olduğu için değil.

### Model hangi soruları görmedi? — sıraya değil, **tarihe** güveniyoruz

Benchmark'ın tüm geçerliliği buna bağlı: test soruları gerçekten eğitimde kullanılmamış
olmalı. "İlk N kayıt eğitimdeydi" gibi bir *varsayım* yeterli değil — sıra değişirse
test sessizce kirlenir ve sonuçlar anlamsız olur.

Bu yüzden ham `dataset.json`'u kullanıyoruz; her kayıtta `answered_at` zaman damgası var.
Eğitim öncesi son veri yüklemesi **2026-07-23T00:12:30**'da yapıldı (HF commit geçmişi).
Bu andan **sonra** üretilen her kayıt, model için kesinlikle yenidir.

İki bağımsız kanıt bu sınırı doğruluyor:

| Kanıt | Sonuç |
|---|---|
| Eğitim logu (`Num examples`) | 757 |
| 00:12:30'da mevcut kayıt sayısı | **757** ✓ |

Aşağıdaki hücre bu tutarlılığı **her çalıştırmada yeniden kontrol eder** — uyuşmazsa uyarır.

In [ ]:
import urllib.request

# Ham veri seti: HF'deki sharegpt surumunde olmayan alanlar burada
# (id, zaman damgasi, tools semasi, senaryo etiketi).
RAW_URL = ("https://raw.githubusercontent.com/BilalAbic/math-toolcall-tr/"
           "main/toolcall-dataset/data/dataset.json")
ham = json.loads(urllib.request.urlopen(RAW_URL).read().decode("utf-8"))
print("Ham veri seti:", len(ham), "kayit")

# Egitim oncesi son HF veri yuklemesinin zamani.
# Bu andan SONRA cevaplanan kayitlari model hic gormedi.
EGITIM_KESIM = "2026-07-23T00:12:30"

egitimde = [r for r in ham if r["answered_at"] <  EGITIM_KESIM]
gorulmemis_hepsi = [r for r in ham if r["answered_at"] >= EGITIM_KESIM]

# --- DOGRULAMA: sinir gercekten tutuyor mu? ---
print(f"\nEgitimde kullanilan : {len(egitimde)}")
print(f"Gorulmemis          : {len(gorulmemis_hepsi)}")

if len(egitimde) == EGITIMDE_KULLANILAN:
    print(f"OK  Egitim logundaki 'Num examples = {EGITIMDE_KULLANILAN}' ile ESLESIYOR.")
else:
    print(f"UYARI! Egitim logu {EGITIMDE_KULLANILAN} diyor ama tarihe gore {len(egitimde)} cikti.")
    print("      EGITIM_KESIM degerini kontrol et - test kirlenmis olabilir.")

# Sira butunlugu: zaman damgalari monoton mu?
_bozuk = sum(1 for a, b in zip(ham, ham[1:]) if a["answered_at"] > b["answered_at"])
print(f"{'OK  Sira korunmus' if _bozuk == 0 else f'UYARI! {_bozuk} kayit sirasiz'}")

# Kimlik cakismasi olmadigini dogrula (ayni soru iki kumede olmasin)
_egitim_id = {r["id"] for r in egitimde}
_cakisma = sum(1 for r in gorulmemis_hepsi if r["id"] in _egitim_id)
print(f"{'OK  Iki kume arasinda cakisma YOK' if _cakisma == 0 else f'UYARI! {_cakisma} cakisma'}")

# Tekrarlanabilir alt kume
import random as _rnd
_rnd.seed(SEED)
gorulmemis = _rnd.sample(gorulmemis_hepsi, min(MATEMATIK_ORNEK_SAYISI, len(gorulmemis_hepsi)))
print(f"\nTest edilecek: {len(gorulmemis)} ornek")

In [ ]:
# --- Modelin ciktisindan arac cagrilarini ayikla ---
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def cagrilari_ayikla(metin: str):
    """Metindeki <tool_call> bloklarini parse eder.
    Doner: (cagri_listesi, format_gecerli_mi)"""
    ham = TOOL_CALL_RE.findall(metin)
    if not ham:
        return [], True          # hic cagri yok -> format acisindan sorun degil
    cagrilar = []
    for parca in ham:
        try:
            cagrilar.append(json.loads(parca))
        except json.JSONDecodeError:
            return cagrilar, False   # bozuk JSON uretmis
    return cagrilar, True


# Ham formatta bunlar dogrudan alan olarak duruyor - regex'e gerek yok.
def beklenen_araclar(ornek):
    return [c.get("name") for c in (ornek.get("tool_calls") or [])]


# --- ADIL PROMPT: iki modele de AYNI bilgiyi ver ---
# Kritik: sadece soruyu gonderirsek base model hangi araclarin var oldugunu ve
# hangi formatta cevap vermesi gerektigini BILEMEZ. O zaman hic arac cagirmaz,
# sadece "arac gerekmiyor" ornekierinden puan toplar ve skoru tamamen yapay olur.
# Bu yuzden araç semalarini ve format talimatini iki modele de veriyoruz.
def prompt_kur(ornek) -> str:
    araclar = [t for t in ornek.get("tools", [])
               if isinstance(t, dict) and isinstance(t.get("function"), dict)]
    satirlar = []
    for t in araclar:
        f = t["function"]
        params = json.dumps(f.get("parameters", {}), ensure_ascii=False)
        satirlar.append(f'- {f["name"]}: {f.get("description","")}\n  parametreler: {params}')
    arac_metni = "\n".join(satirlar)

    return (
        "Kullanabileceğin araçlar:\n"
        f"{arac_metni}\n\n"
        "Bir araç kullanman gerekiyorsa TAM OLARAK şu formatta yaz:\n"
        '<tool_call>{"name": "araç_adı", "arguments": {"parametre": "değer"}}</tool_call>\n'
        "Birden fazla araç gerekiyorsa her birini ayrı <tool_call> bloğunda yaz.\n"
        "Araç gerekmiyorsa veya zorunlu bir bilgi eksikse araç çağırma, doğrudan yanıt ver.\n\n"
        f"Soru: {ornek['question']}"
    )


# Ornek kontrol
_o = gorulmemis[0]
print("SENARYO  :", _o["scenario"], "|", _o["domain"])
print("BEKLENEN :", beklenen_araclar(_o) or "(arac cagrilmamali)")
print("\n--- MODELE GIDEN PROMPT ---")
print(prompt_kur(_o)[:700])

In [ ]:
# Promptlari bir kere kur
mat_promptlar = [prompt_kur(o) for o in gorulmemis]


def matematik_calistir(model_adi: str) -> dict:
    n = len(gorulmemis)
    basla = time.time()
    print(f"[{model_adi}] Matematik tool-call uretiliyor...")

    # max_new_tokens=256: <think> + <tool_call> JSON'u sigacak kadar
    with model_olarak(model_adi):
        ciktilar = uret_toplu(mat_promptlar, max_new_tokens=256)

    format_ok = arac_ok = abstain_ok = 0
    abstain_toplam = cagri_toplam = 0
    kayitlar = []

    for i, cikti in enumerate(ciktilar):
        ornek = gorulmemis[i]
        beklenen = beklenen_araclar(ornek)
        uretilen, gecerli = cagrilari_ayikla(cikti)
        uretilen_isimler = [c.get("name") for c in uretilen]

        if gecerli:
            format_ok += 1

        if beklenen:                      # arac CAGRILMALI olan ornek
            cagri_toplam += 1
            # kume karsilastirmasi: sira onemli degil, dogru araclar cagrildi mi
            if set(uretilen_isimler) == set(beklenen):
                arac_ok += 1
        else:                             # arac cagrilMAMALI olan ornek
            abstain_toplam += 1
            if not uretilen_isimler:
                abstain_ok += 1

        kayitlar.append({
            "no": i, "model": model_adi, "senaryo": ornek["scenario"],
            "beklenen": beklenen, "uretilen": uretilen_isimler,
            "format_gecerli": gecerli,
            "cikti": cikti[:300],
        })

    print(f"[{model_adi}] genel {100*(arac_ok+abstain_ok)/n:.1f}% "
          f"({time.time()-basla:.0f}s)")

    return {
        "model": model_adi,
        "format_gecerlilik": round(100 * format_ok / n, 2),
        "arac_secim_dogrulugu": round(100 * arac_ok / cagri_toplam, 2) if cagri_toplam else None,
        "abstain_dogrulugu": round(100 * abstain_ok / abstain_toplam, 2) if abstain_toplam else None,
        "genel_dogruluk": round(100 * (arac_ok + abstain_ok) / n, 2),
        "cagri_gereken": cagri_toplam,
        "cagri_gerekmeyen": abstain_toplam,
        "sure_sn": round(time.time() - basla, 1),
        "kayitlar": kayitlar,
    }


mat_sonuc = {ad: matematik_calistir(ad) for ad in ("base", "finetune")}
for ad, s in mat_sonuc.items():
    print(f"{ad:9} -> genel %{s['genel_dogruluk']} | arac %{s['arac_secim_dogrulugu']} | "
          f"abstain %{s['abstain_dogrulugu']} | format %{s['format_gecerlilik']}")

## 7. Benchmark 3 — GSM8K (sektör standardı)

[`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k) — ilkokul seviyesi matematik
problemleri, çok adımlı akıl yürütme gerektirir. Dil modeli literatüründe **en çok atıf
alan matematik benchmark'ı** (889 bin indirme).

Neden notlandırması güvenilir: her cevap `#### 18` biçiminde **kesin bir sayıyla** biter.
Anlamsal benzerliğe gerek yok, sayı tutuyor mu diye bakıyoruz.

> ⚠️ **Dil farkı — dürüst olalım.** GSM8K **İngilizce**, bizim fine-tune ise **Türkçe**
> veriyle eğitildi. Burada fine-tune'un *kazandırması* beklenmiyor. Ölçtüğümüz şey:
> Türkçe tool-call eğitimi modelin **genel matematik yeteneğini bozdu mu?**
> Skor korunuyorsa fine-tune "temiz" demektir.

Ek olarak bir yan metrik topluyoruz: **gereksiz `<tool_call>` oranı.** Fine-tune, düz
matematik sorusuna araç çağrısıyla cevap veriyorsa formata aşırı uyum sağlamış demektir —
bu, sayısal skorda görünmeyen önemli bir yan etkidir.

In [ ]:
from datasets import load_dataset   # ham veri setini urllib ile aldik, bunu simdi kullaniyoruz

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
gsm8k = gsm8k.shuffle(seed=SEED).select(range(min(GSM8K_SORU_SAYISI, len(gsm8k))))
print("GSM8K test ornegi:", len(gsm8k))
print("\nORNEK SORU:", gsm8k[0]["question"][:200])
print("ORNEK CEVAP:", repr(gsm8k[0]["answer"][-60:]))

In [ ]:
def son_sayiyi_al(metin: str):
    """Metindeki SON sayiyi dondurur. Model genelde sonucu en sona yazar."""
    # Binlik ayraci, para birimi ve yuzde isaretlerini temizle: '1,234.5' -> '1234.5'
    temiz = metin.replace(",", "").replace("$", "").replace("%", "")
    sayilar = re.findall(r"-?\d+\.?\d*", temiz)
    if not sayilar:
        return None
    try:
        return float(sayilar[-1])
    except ValueError:
        return None


def gsm8k_dogru_cevap(cevap_metni: str):
    """'#### 18' kalibindaki altin cevabi sayiya cevirir."""
    return son_sayiyi_al(cevap_metni.split("####")[-1])


# Promptlari bir kere kur
gsm_promptlar = [
    o["question"] + "\n\nSolve step by step, then give the final numeric answer on the last line."
    for o in gsm8k
]


def gsm8k_calistir(model_adi: str) -> dict:
    n = len(gsm8k)
    basla = time.time()
    print(f"[{model_adi}] GSM8K uretiliyor...")

    # max_new_tokens=320: cok adimli akil yurutme icin yer birak.
    # Cok kisarsan model sonuca ulasamadan kesilir -> haksiz dusuk skor.
    with model_olarak(model_adi):
        ciktilar = uret_toplu(gsm_promptlar, max_new_tokens=320)

    dogru = arac_cagirdi = 0
    kayitlar = []
    for i, cikti in enumerate(ciktilar):
        tahmin = son_sayiyi_al(cikti)
        altin = gsm8k_dogru_cevap(gsm8k[i]["answer"])
        # Kayan nokta karsilastirmasi: kucuk tolerans birak (18 vs 18.0)
        sonuc = tahmin is not None and altin is not None and abs(tahmin - altin) < 1e-4
        if sonuc:
            dogru += 1

        # Yan metrik: duz matematik sorusuna arac cagirmaya calisti mi?
        cagrilar, _ = cagrilari_ayikla(cikti)
        if cagrilar:
            arac_cagirdi += 1

        kayitlar.append({"no": i, "model": model_adi, "tahmin": tahmin,
                         "altin": altin, "dogru_mu": sonuc,
                         "arac_cagirdi": bool(cagrilar), "cikti": cikti[-200:]})

    print(f"[{model_adi}] dogru {dogru}/{n} = {100*dogru/n:.1f}% ({time.time()-basla:.0f}s)")

    return {
        "model": model_adi,
        "dogru": dogru,
        "toplam": n,
        "basari": round(100 * dogru / n, 2),
        "gereksiz_arac_cagrisi": round(100 * arac_cagirdi / n, 2),
        "sure_sn": round(time.time() - basla, 1),
        "kayitlar": kayitlar,
    }


gsm_sonuc = {ad: gsm8k_calistir(ad) for ad in ("base", "finetune")}
for ad, s in gsm_sonuc.items():
    print(f"{ad:9} -> %{s['basari']} ({s['dogru']}/{s['toplam']}) | "
          f"gereksiz arac cagrisi: %{s['gereksiz_arac_cagrisi']}")

## 8. Karşılaştırma

In [ ]:
def fark(yeni, eski):
    if yeni is None or eski is None:
        return "-"
    d = yeni - eski
    return f"{d:+.2f}"

ozet = pd.DataFrame([
    {"Benchmark": "MMLU (genel bilgi)", "Metrik": "Başarı %",
     "Base": mmlu_sonuc['base']['basari'], "Fine-tune": mmlu_sonuc['finetune']['basari'],
     "Fark": fark(mmlu_sonuc['finetune']['basari'], mmlu_sonuc['base']['basari'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Genel doğruluk %",
     "Base": mat_sonuc['base']['genel_dogruluk'], "Fine-tune": mat_sonuc['finetune']['genel_dogruluk'],
     "Fark": fark(mat_sonuc['finetune']['genel_dogruluk'], mat_sonuc['base']['genel_dogruluk'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Araç seçimi %",
     "Base": mat_sonuc['base']['arac_secim_dogrulugu'], "Fine-tune": mat_sonuc['finetune']['arac_secim_dogrulugu'],
     "Fark": fark(mat_sonuc['finetune']['arac_secim_dogrulugu'], mat_sonuc['base']['arac_secim_dogrulugu'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Çekimserlik (abstain) %",
     "Base": mat_sonuc['base']['abstain_dogrulugu'], "Fine-tune": mat_sonuc['finetune']['abstain_dogrulugu'],
     "Fark": fark(mat_sonuc['finetune']['abstain_dogrulugu'], mat_sonuc['base']['abstain_dogrulugu'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Format geçerliliği %",
     "Base": mat_sonuc['base']['format_gecerlilik'], "Fine-tune": mat_sonuc['finetune']['format_gecerlilik'],
     "Fark": fark(mat_sonuc['finetune']['format_gecerlilik'], mat_sonuc['base']['format_gecerlilik'])},
    {"Benchmark": "GSM8K (standart)", "Metrik": "Başarı %",
     "Base": gsm_sonuc['base']['basari'], "Fine-tune": gsm_sonuc['finetune']['basari'],
     "Fark": fark(gsm_sonuc['finetune']['basari'], gsm_sonuc['base']['basari'])},
    {"Benchmark": "GSM8K (standart)", "Metrik": "Gereksiz araç çağrısı % (düşük iyi)",
     "Base": gsm_sonuc['base']['gereksiz_arac_cagrisi'], "Fine-tune": gsm_sonuc['finetune']['gereksiz_arac_cagrisi'],
     "Fark": fark(gsm_sonuc['finetune']['gereksiz_arac_cagrisi'], gsm_sonuc['base']['gereksiz_arac_cagrisi'])},
])
display(ozet)

print("\nNASIL OKUNUR")
print("  MMLU farki ~0 veya pozitif   -> genel bilgi korunmus (istedigimiz bu)")
print("  GSM8K farki ~0 veya pozitif  -> matematik akil yurutme korunmus")
print("  Ikisi de cok negatif         -> asiri uyum, model daralmis")
print("  Matematik tool-call pozitif  -> fine-tune ise yaramis")
print("  Gereksiz arac cagrisi YUKSEK -> formata asiri uyum; duz soruya arac cagiriyor")

In [ ]:
import matplotlib.pyplot as plt

metrikler = ozet['Metrik'].tolist()
base_d = [0 if v is None else v for v in ozet['Base']]
ft_d   = [0 if v is None else v for v in ozet['Fine-tune']]

x = range(len(metrikler)); g = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - g/2 for i in x], base_d, g, label='Base', color='#94a3b8')
ax.bar([i + g/2 for i in x], ft_d,   g, label='Fine-tune', color='#2563eb')

for i, (b, f) in enumerate(zip(base_d, ft_d)):
    ax.text(i - g/2, b + 1, f"{b:.1f}", ha='center', fontsize=9)
    ax.text(i + g/2, f + 1, f"{f:.1f}", ha='center', fontsize=9)

ax.set_xticks(list(x)); ax.set_xticklabels(metrikler, rotation=18, ha='right')
ax.set_ylabel('%'); ax.set_ylim(0, 105)
ax.set_title('Base vs Fine-tune')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

### Örnek çıktılar

Sayılar *ne kadar* değiştiğini söyler; asıl anlamak için çıktılara bakmak gerekir.

In [ ]:
# Fine-tune'un DOGRU, base'in YANLIS yaptigi ornekler -> fine-tune tam olarak ne kazandirdi
b_kayit = {k['no']: k for k in mat_sonuc['base']['kayitlar']}
f_kayit = {k['no']: k for k in mat_sonuc['finetune']['kayitlar']}

gosterildi = 0
for no, f in f_kayit.items():
    b = b_kayit[no]
    f_dogru = set(f['uretilen']) == set(f['beklenen'])
    b_dogru = set(b['uretilen']) == set(b['beklenen'])
    if f_dogru and not b_dogru:
        print("=" * 78)
        print("BEKLENEN :", f['beklenen'])
        print("BASE     :", b['uretilen'], "|", b['cikti'][:180].replace("\n", " "))
        print("FINETUNE :", f['uretilen'], "|", f['cikti'][:180].replace("\n", " "))
        gosterildi += 1
        if gosterildi >= 3:
            break

if gosterildi == 0:
    print("Fine-tune'un base'e ustunluk sagladigi ornek bulunamadi.")

In [ ]:
# Aksi yon: fine-tune'un BOZDUGU ornekler (varsa) -> durust degerlendirme icin sart
gosterildi = 0
for no, f in f_kayit.items():
    b = b_kayit[no]
    f_dogru = set(f['uretilen']) == set(f['beklenen'])
    b_dogru = set(b['uretilen']) == set(b['beklenen'])
    if b_dogru and not f_dogru:
        print("=" * 78)
        print("BEKLENEN :", f['beklenen'])
        print("BASE     :", b['uretilen'], "(dogru)")
        print("FINETUNE :", f['uretilen'], "(yanlis) |", f['cikti'][:180].replace("\n", " "))
        gosterildi += 1
        if gosterildi >= 3:
            break

if gosterildi == 0:
    print("Fine-tune hicbir ornegi bozmamis.")

## 9. Sonuçları kaydet

In [ ]:
# Ozet tablo
ozet.to_csv("benchmark_ozet.csv", index=False)

# Tum ham kayitlar (hangi soruya ne cevap verildi)
tum = []
for kaynak, sonuclar in (("mmlu", mmlu_sonuc), ("matematik", mat_sonuc), ("gsm8k", gsm_sonuc)):
    for ad, s in sonuclar.items():
        for k in s["kayitlar"]:
            tum.append({"benchmark": kaynak, **k})
pd.DataFrame(tum).to_csv("benchmark_detay.csv", index=False)

# Bolum bazli MMLU karsilastirmasi
bolum_df = pd.DataFrame({
    "base":     mmlu_sonuc['base']['bolum_basari'],
    "finetune": mmlu_sonuc['finetune']['bolum_basari'],
}).fillna(0)
bolum_df["fark"] = bolum_df["finetune"] - bolum_df["base"]
bolum_df = bolum_df.sort_values("fark")
bolum_df.to_csv("benchmark_mmlu_bolum.csv")

print("Kaydedildi: benchmark_ozet.csv, benchmark_detay.csv, benchmark_mmlu_bolum.csv")
print("\nMMLU'da en cok DUSEN 5 bolum:"); display(bolum_df.head(5))
print("MMLU'da en cok YUKSELEN 5 bolum:"); display(bolum_df.tail(5))

---

### Not: `alibayram/*` veri setlerine yazma yok

Orijinal kodda sonuçlar `alibayram/...` repolarına `push_to_hub` ile gönderiliyordu.
O repolar **başkasına ait** — oraya yazmak doğru olmaz. Bu notebook sonuçları yalnızca
yerel CSV olarak kaydeder. Kendi hesabına yüklemek istersen:

```python
from datasets import Dataset
Dataset.from_pandas(ozet).push_to_hub("bilalabic/math-toolcall-benchmark", token=HF_TOKEN)
```

### Sonuçları nasıl yorumlamalı

| Gözlem | Anlamı | Ne yapmalı |
|---|---|---|
| MMLU / GSM8K'da büyük düşüş | Dar alana aşırı uyum (catastrophic forgetting) | Epoch'u 2'ye, LR'yi 1e-4'e düşür; veriye genel amaçlı örnek karıştır |
| Matematik tool-call'da kazanç yok | LoRA kapasitesi yetersiz olabilir | `r=16` (alpha 16) dene ya da veri setini büyüt |
| Format %100 ama araç seçimi düşük | Formatı öğrenmiş, *hangi* aracı seçeceğini öğrenememiş | Çeldirici araçlar fazla zor olabilir; veri dengesini gözden geçir |
| Gereksiz araç çağrısı yüksek | Formata aşırı uyum — düz soruya bile araç çağırıyor | `arac_gereksiz` senaryosunun payını artır |
| GSM8K düşük ama tool-call yüksek | Beklenen sonuç: model *çağırmayı* öğrendi, *hesaplamayı* değil | Sorun değil — model zaten aracı çağırıp sonucu yorumlasın diye eğitildi |

### İdeal tablo neye benzer

```
MMLU              : base ≈ finetune        (bozulma yok)
GSM8K             : base ≈ finetune        (matematik yeteneği korundu)
Tool-call genel   : finetune >> base       (asıl kazanç)
Gereksiz çağrı    : finetune ≈ base        (formata saplanmamış)
```